# Uncertainty Quantification with Kernel Density Matrices

This notebook demonstrates how **KDMClassModel** naturally quantifies predictive
uncertainty through its probability-distribution outputs, and contrasts it with a
conventional **softmax classifier** built on the same CNN encoder.

**Experiment:** We apply controlled rotations (0°–345° in 15° steps) to MNIST test
images of digits **'1'** and **'6'** and measure the **Shannon entropy** of the
predicted class distribution at each rotation angle.

Softmax classifiers are notoriously **overconfident** on out-of-distribution inputs —
they maintain high-confidence predictions even for heavily rotated digits. KDM
classifiers become appropriately **uncertain**: when the input falls far from all
learned prototypes, the kernel weights decay to near-uniform, and the output
distribution approaches the uniform (high entropy). No temperature scaling or
post-hoc calibration is needed.

PyTorch implementation of the example described in:
*Kernel Density Matrices for Probabilistic Deep Learning* (González et al., QMI 2025).

In [ ]:
# Uncomment to install the required packages
# !pip install git+https://github.com/fagonzalezo/kdm.git

## 1. Imports and Setup

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF

from kdm.models import KDMClassModel
from kdm.init import init_kdm_layer

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
torch.manual_seed(42)
np.random.seed(42)

## 2. Dataset Loading

We use a 20% subset of MNIST training data (~12 000 samples) for a quick demo.
Remove the index slice to use the full 60 000 samples for better accuracy.

In [ ]:
mnist_train_full = datasets.MNIST(
    root='~/.cache/mnist', train=True, download=True,
    transform=transforms.ToTensor()
)
mnist_test_ds = datasets.MNIST(
    root='~/.cache/mnist', train=False, download=True,
    transform=transforms.ToTensor()
)

# 20% subset — remove the slice below to train on the full 60k.
perm = np.random.RandomState(42).permutation(len(mnist_train_full))
train_idx = perm[:12000]

def stack(ds, idx):
    xs = torch.stack([ds[i][0] for i in idx])
    ys = torch.tensor([ds[i][1] for i in idx], dtype=torch.long)
    return xs, ys

X_train, y_train = stack(mnist_train_full, train_idx)
X_test  = torch.stack([mnist_test_ds[i][0] for i in range(len(mnist_test_ds))])
y_test  = torch.tensor([mnist_test_ds[i][1] for i in range(len(mnist_test_ds))],
                       dtype=torch.long)

print(f"train={X_train.shape}  test={X_test.shape}")

## 3. Model Architectures

Both models share the **same CNN encoder** architecture that maps 28×28 grayscale
images to a 32-dimensional latent vector. The only difference is the **output head**:

| Model | Head | Loss |
|---|---|---|
| Softmax baseline | `Linear(32, 10)` | cross-entropy |
| KDMClassModel | `KDMLayer` + `dm2discrete` | NLL on KDM output |

Using the same encoder isolates the effect of the output head on uncertainty.

In [ ]:
ENCODED_SIZE = 32
BASE_DEPTH   = 32
DIM_Y        = 10
N_COMP       = 100
BATCH_SIZE   = 128


def create_encoder(base_depth: int, encoded_size: int) -> nn.Module:
    """Small CNN: (bs,1,28,28) -> (bs, encoded_size)."""
    return nn.Sequential(
        nn.Conv2d(1, base_depth, 5, stride=1, padding=2),              nn.LeakyReLU(),
        nn.Conv2d(base_depth, base_depth, 5, stride=2, padding=2),     nn.LeakyReLU(),
        nn.Conv2d(base_depth, 2*base_depth, 5, stride=1, padding=2),   nn.LeakyReLU(),
        nn.Conv2d(2*base_depth, 2*base_depth, 5, stride=2, padding=2), nn.LeakyReLU(),
        nn.Conv2d(2*base_depth, 4*encoded_size, 7, stride=1, padding=0), nn.LeakyReLU(),
        nn.Flatten(),
        nn.Linear(4*encoded_size, encoded_size),
    )


@torch.no_grad()
def evaluate_base(model, X, y, batch_size=512):
    """Evaluate baseline (logit output) on tensors X, y."""
    model.eval()
    total_loss, n_correct = 0.0, 0
    for i in range(0, len(X), batch_size):
        xb, yb = X[i:i+batch_size], y[i:i+batch_size]
        logits = model(xb)
        total_loss += F.cross_entropy(logits, yb).item() * xb.size(0)
        n_correct  += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(X), n_correct / len(X)


@torch.no_grad()
def evaluate_kdm(model, X, y, batch_size=512):
    """Evaluate KDMClassModel (probability output) on tensors X, y."""
    model.eval()
    total_loss, n_correct = 0.0, 0
    for i in range(0, len(X), batch_size):
        xb, yb = X[i:i+batch_size], y[i:i+batch_size]
        probs = model(xb)
        total_loss += F.nll_loss(torch.log(probs.clamp_min(1e-12)), yb).item() * xb.size(0)
        n_correct  += (probs.argmax(1) == yb).sum().item()
    return total_loss / len(X), n_correct / len(X)

## 4. Softmax Baseline

A conventional deep classifier: CNN encoder + linear head + softmax.

In [ ]:
encoder_base   = create_encoder(BASE_DEPTH, ENCODED_SIZE)
baseline_model = nn.Sequential(encoder_base, nn.Linear(ENCODED_SIZE, DIM_Y))

opt_base  = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)
loader    = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
hist_base = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

EPOCHS_BASE = 5

for ep in range(EPOCHS_BASE):
    baseline_model.train()
    for xb, yb in loader:
        loss = F.cross_entropy(baseline_model(xb), yb)
        opt_base.zero_grad(); loss.backward(); opt_base.step()
    tr_loss, tr_acc = evaluate_base(baseline_model, X_train, y_train)
    te_loss, te_acc = evaluate_base(baseline_model, X_test,  y_test)
    for k, v in [('train_loss', tr_loss), ('test_loss', te_loss),
                 ('train_acc',  tr_acc),  ('test_acc',  te_acc)]:
        hist_base[k].append(v)
    print(f"epoch {ep+1:2d}  train={tr_loss:.4f}/{tr_acc:.3f}  test={te_loss:.4f}/{te_acc:.3f}")

## 5. KDM Classifier

We initialise the KDM encoder from a **deepcopy** of the trained baseline encoder,
giving both models the same representational starting point. Only the output head
(KDM layer + kernel) is different.

In [ ]:
encoder_kdm = copy.deepcopy(encoder_base)   # start from baseline's trained weights
kdm_model   = KDMClassModel(
    encoded_size=ENCODED_SIZE, dim_y=DIM_Y,
    encoder=encoder_kdm, n_comp=N_COMP, sigma=1.0, w_train=True,
)

# Initialise KDM prototypes from a random subset of training encodings.
idx = np.random.randint(X_train.shape[0], size=N_COMP)
encoder_kdm.eval()
with torch.no_grad():
    enc_x = encoder_kdm(X_train[idx])
init_kdm_layer(
    kdm_model.kdm,
    encoded_x=enc_x,
    samples_y=F.one_hot(y_train[idx], DIM_Y).float(),
    init_sigma=True,
)
print(f"Initial sigma: {float(kdm_model.kernel.sigma):.4f}")

opt_kdm  = torch.optim.Adam(kdm_model.parameters(), lr=1e-3)
hist_kdm = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

EPOCHS_KDM = 5

for ep in range(EPOCHS_KDM):
    kdm_model.train()
    for xb, yb in loader:
        probs = kdm_model(xb)
        loss  = F.nll_loss(torch.log(probs.clamp_min(1e-12)), yb)
        opt_kdm.zero_grad(); loss.backward(); opt_kdm.step()
    tr_loss, tr_acc = evaluate_kdm(kdm_model, X_train, y_train)
    te_loss, te_acc = evaluate_kdm(kdm_model, X_test,  y_test)
    for k, v in [('train_loss', tr_loss), ('test_loss', te_loss),
                 ('train_acc',  tr_acc),  ('test_acc',  te_acc)]:
        hist_kdm[k].append(v)
    print(f"epoch {ep+1:2d}  train={tr_loss:.4f}/{tr_acc:.3f}  "
          f"test={te_loss:.4f}/{te_acc:.3f}  sigma={float(kdm_model.kernel.sigma):.4f}")

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))

axes[0].plot(hist_base['train_loss'], label='Baseline train', color='C1')
axes[0].plot(hist_base['test_loss'],  label='Baseline test',  color='C1', linestyle='--')
axes[0].plot(hist_kdm['train_loss'],  label='KDM train',      color='C0')
axes[0].plot(hist_kdm['test_loss'],   label='KDM test',       color='C0', linestyle='--')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('epoch')

axes[1].plot(hist_base['train_acc'], label='Baseline train', color='C1')
axes[1].plot(hist_base['test_acc'],  label='Baseline test',  color='C1', linestyle='--')
axes[1].plot(hist_kdm['train_acc'],  label='KDM train',      color='C0')
axes[1].plot(hist_kdm['test_acc'],   label='KDM test',       color='C0', linestyle='--')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('epoch')

plt.tight_layout()
plt.show()

_, base_te_acc = evaluate_base(baseline_model, X_test, y_test)
_, kdm_te_acc  = evaluate_kdm(kdm_model,       X_test, y_test)
print(f"Baseline test accuracy : {base_te_acc:.4f}")
print(f"KDM      test accuracy : {kdm_te_acc:.4f}")

## 7. Rotation Experiment

We select one test image of digit **'1'** and one of **'6'**:

- **'1'** is nearly rotationally symmetric → we expect **low, flat** uncertainty.
- **'6'** is asymmetric: at ~180° it resembles a **'9'** → uncertainty should spike
  and the model should hedge between classes 6 and 9.

For each digit we generate 24 rotated copies (0°, 15°, …, 345°) using
`torchvision.transforms.functional.rotate`, which applies bilinear interpolation
and fills out-of-bounds pixels with black (matching the MNIST background).

In [ ]:
def first_of_digit(X, y, digit):
    """Return the first test image of the given digit as (1,1,28,28)."""
    idx = (y == digit).nonzero(as_tuple=True)[0][0]
    return X[idx:idx+1]

img_1 = first_of_digit(X_test, y_test, 1)
img_6 = first_of_digit(X_test, y_test, 6)

fig, axes = plt.subplots(1, 2, figsize=(3, 1.5))
axes[0].imshow(img_1[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0].set_title("digit '1'"); axes[0].axis('off')
axes[1].imshow(img_6[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[1].set_title("digit '6'"); axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
ANGLES = list(range(0, 360, 15))   # 24 angles: 0, 15, ..., 345


def rotate_batch(img, angles):
    """Rotate a (1,1,H,W) image by each angle. Returns (len(angles),1,H,W)."""
    return torch.stack([TF.rotate(img[0], float(a), fill=0.0) for a in angles])


@torch.no_grad()
def predict_rotations(model, img, angles, is_logit=False):
    """
    Run model on all rotations of img.

    Args:
        is_logit: True for the softmax baseline (output needs softmax),
                  False for KDMClassModel (already returns probabilities).

    Returns:
        probs   (n_angles, 10)  -- class probabilities
        entropy (n_angles,)     -- Shannon entropy H(p)
        conf    (n_angles,)     -- max probability (confidence)
    """
    model.eval()
    rotated = rotate_batch(img, angles)      # (24, 1, 28, 28)
    out = model(rotated)
    probs = F.softmax(out, dim=1).cpu().numpy() if is_logit else out.cpu().numpy()
    eps = 1e-12
    entropy = -(probs * np.log(probs + eps)).sum(axis=1)
    conf    = probs.max(axis=1)
    return probs, entropy, conf


# Run predictions for both models and both digits.
kdm_probs_1,  kdm_entropy_1,  kdm_conf_1  = predict_rotations(kdm_model,      img_1, ANGLES)
kdm_probs_6,  kdm_entropy_6,  kdm_conf_6  = predict_rotations(kdm_model,      img_6, ANGLES)
base_probs_1, base_entropy_1, base_conf_1 = predict_rotations(baseline_model, img_1, ANGLES, is_logit=True)
base_probs_6, base_entropy_6, base_conf_6 = predict_rotations(baseline_model, img_6, ANGLES, is_logit=True)

## 8. Visualisation

In [ ]:
# Image strip: show both digits at 0°, 90°, 180°, 270°.
showcase_angles = [0, 90, 180, 270]

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for row, (img, label) in enumerate([(img_1, "Digit '1'"), (img_6, "Digit '6'")]):
    frames = rotate_batch(img, showcase_angles)
    for col, (ax, angle) in enumerate(zip(axes[row], showcase_angles)):
        ax.imshow(frames[col, 0].numpy(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f"{label}\n{angle}\u00b0", fontsize=8)
        ax.axis('off')
plt.suptitle("Rotated test images", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Uncertainty curves: KDM vs. Baseline, one panel per digit.
# Solid lines = entropy (left y-axis), dashed lines = confidence (right y-axis).

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

max_entropy = np.log(10)   # theoretical maximum for 10 classes (uniform distribution)

for ax, (kdm_h, base_h, kdm_c, base_c, label) in zip(axes, [
    (kdm_entropy_1, base_entropy_1, kdm_conf_1, base_conf_1, "Digit '1'"),
    (kdm_entropy_6, base_entropy_6, kdm_conf_6, base_conf_6, "Digit '6'"),
]):
    l1, = ax.plot(ANGLES, kdm_h,  color='steelblue', marker='o', ms=4,
                  label='KDM \u2014 entropy')
    l2, = ax.plot(ANGLES, base_h, color='tomato',    marker='s', ms=4,
                  label='Baseline \u2014 entropy')
    ax.axhline(max_entropy, color='gray', linestyle=':', alpha=0.5,
               label='max entropy (uniform)')

    ax2 = ax.twinx()
    l3, = ax2.plot(ANGLES, kdm_c,  color='steelblue', linestyle='--', alpha=0.6,
                   label='KDM \u2014 confidence')
    l4, = ax2.plot(ANGLES, base_c, color='tomato',    linestyle='--', alpha=0.6,
                   label='Baseline \u2014 confidence')
    ax2.set_ylabel('Max probability (confidence)')
    ax2.set_ylim(0, 1.05)

    ax.set_xlabel('Rotation angle (degrees)')
    ax.set_ylabel('Entropy H(p)')
    ax.set_title(f"Uncertainty under rotation \u2014 {label}")
    ax.set_xticks(ANGLES[::2])
    ax.set_xlim(-5, 365)

    all_lines  = [l1, l2, l3, l4]
    all_labels = [l.get_label() for l in all_lines]
    ax.legend(all_lines, all_labels, loc='upper left', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Probability heatmaps: 2 rows (digit) x 2 columns (model).
# Each cell shows P(class | rotated image) as a colour map.

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

data = [
    (kdm_probs_1,  "KDM \u2014 Digit '1'",       0, 0),
    (base_probs_1, "Baseline \u2014 Digit '1'",   0, 1),
    (kdm_probs_6,  "KDM \u2014 Digit '6'",       1, 0),
    (base_probs_6, "Baseline \u2014 Digit '6'",   1, 1),
]

for probs, title, r, c in data:
    ax = axes[r, c]
    # probs: (n_angles, 10) -- transpose so rows=class, cols=angle
    im = ax.imshow(probs.T, aspect='auto', interpolation='nearest',
                   cmap='viridis', vmin=0, vmax=1, origin='upper')
    ax.set_yticks(range(10))
    ax.set_yticklabels([str(i) for i in range(10)])
    ax.set_xticks(range(0, len(ANGLES), 2))
    ax.set_xticklabels([str(ANGLES[i]) for i in range(0, len(ANGLES), 2)],
                       rotation=45, ha='right', fontsize=7)
    ax.set_xlabel('Rotation angle (degrees)')
    ax.set_ylabel('Class')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='P(class)')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical summary: predicted class and entropy at each angle.
header = (f"{'Angle':>6}  "
          f"{'KDM pred-1':>10}  {'KDM H-1':>8}  "
          f"{'Base pred-1':>11}  {'Base H-1':>9}  "
          f"{'KDM pred-6':>10}  {'KDM H-6':>8}  "
          f"{'Base pred-6':>11}  {'Base H-6':>9}")
print(header)
print("-" * len(header))
for i, angle in enumerate(ANGLES):
    print(f"{angle:>6}\u00b0   "
          f"{kdm_probs_1[i].argmax():>8}     {kdm_entropy_1[i]:>6.3f}    "
          f"{base_probs_1[i].argmax():>9}     {base_entropy_1[i]:>7.3f}    "
          f"{kdm_probs_6[i].argmax():>8}     {kdm_entropy_6[i]:>6.3f}    "
          f"{base_probs_6[i].argmax():>9}     {base_entropy_6[i]:>7.3f}")

## 9. Discussion

### What to observe

**Softmax baseline (overconfidence):**
Across all rotation angles, the baseline's entropy stays low and its confidence
remains close to 1.0 — even when the digit has been rotated 90° or 180° and no
longer resembles a canonical MNIST digit. This is the well-known *overconfidence*
problem of softmax classifiers: the softmax normalisation always produces a peaked
distribution, regardless of how far the input is from the training distribution.

**KDM classifier (calibrated uncertainty):**
- *Digit '1':* Entropy is low near 0° and 180° (a '1' looks similar upright and
  inverted), but rises noticeably around 90° and 270° where the digit resembles a
  horizontal stroke with no counterpart in MNIST.
- *Digit '6':* Entropy rises sharply around 90°–135° where the rotated shape is
  ambiguous, and again near 180° — where the heatmap clearly shows the model
  shifting probability mass from class 6 toward class 9 (a '6' upside-down is a '9').

### Why KDM naturally quantifies uncertainty

The KDM forward pass computes:

$$P(y \mid x) \propto \sum_k w_k \, \kappa(x, c_k^x)^2 \, [c_k^y]$$

where $\kappa$ is the RBF kernel and $c_k^x$ are learned prototype locations.
When the input $x$ is *out-of-distribution*, all kernel evaluations
$\kappa(x, c_k^x) \approx 0$, the weights collapse toward a near-uniform
distribution, and entropy rises accordingly. **No temperature scaling,
confidence calibration, or ensembling is required.** The uncertainty is an
intrinsic property of the model's architecture.

### Uncertainty measures used

| Measure | Formula | Range | Interpretation |
|---|---|---|---|
| Shannon entropy | $H = -\sum_c p_c \log p_c$ | $[0,\, \log 10 \approx 2.30]$ | Overall distributional uncertainty |
| Max probability (confidence) | $\max_c p_c$ | $[0.1,\, 1.0]$ | How sure is the top prediction? |